# Chapter 5 — PSCI: Power State Coordination Interface

## Concept
PSCI (Power State Coordination Interface) ARM DEN0022 spec defines
CPU_ON / CPU_OFF / CPU_SUSPEND / SYSTEM_RESET / SYSTEM_OFF calls.
The calling convention uses SMC (EL3) or HVC (EL2) — configured by the
`psci-method` Device Tree / ACPI property. QEMU emulates PSCI via the
`-machine virt` PSCI conduit (HVC by default). SMP secondary cores are
brought online by the primary via CPU_ON.

## Lab Objectives
1. Boot a 4-vCPU VM; confirm all 4 secondaries reach online state.
2. Verify all 4 CPUs visible in `/proc/cpuinfo` and `/sys/devices/system/cpu/`.
3. Issue `system_reset` via QMP; VM must reboot and reach login again.
4. Confirm PSCI method (SMC or HVC) from dmesg.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 4
# 4-vCPU boot; remove -no-reboot for system_reset test
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=["-netdev", "user,id=net0", "-device", "virtio-net-pci,netdev=net0"],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Query vCPU count via QMP ─────────────────────────────────────────
import json
cpus = qmp.query_cpus()
print(f"QMP reports {len(cpus)} vCPU(s):")
for c in cpus:
    print(f"  Index {c['cpu-index']}  thread_id={c.get('thread-id','?')}")


In [ ]:
# ── Step 2: Confirm CPUs online in the guest ─────────────────────────────────
cpuinfo = sc.run_command("grep 'processor' /proc/cpuinfo | wc -l", timeout=10)
print(f"Guest /proc/cpuinfo processor count: {cpuinfo.strip()}")

cpu_dirs = sc.run_command(
    "ls /sys/devices/system/cpu/ | grep -E '^cpu[0-9]+$' | sort",
    timeout=10
)
print("CPU directories in sysfs:\n", cpu_dirs)


In [ ]:
# ── Step 3: Read PSCI method from dmesg ──────────────────────────────────────
psci_dmesg = sc.run_command("dmesg | grep -i 'psci'", timeout=10)
print("PSCI dmesg lines:\n", psci_dmesg)


In [ ]:
# ── Step 4: Issue system_reset via QMP — VM must reboot ──────────────────────
print("Issuing QMP system_reset …")
reset_start = time.monotonic()
try:
    qmp.system_reset()
    print("system_reset command sent")
except Exception as e:
    # Socket may close during reset — this is expected
    print(f"Note: {e} (normal if QEMU closes QMP socket during reset)")

# Wait for reboot — serial console goes away and comes back
print("Waiting for guest reboot (up to 120 s) …")
time.sleep(5)   # Brief pause before reconnect attempt

# Reconnect QMP after reset
import socket as _sock
deadline = time.monotonic() + 120
reconnected = False
while time.monotonic() < deadline:
    try:
        qmp2 = QMPClient(port=launcher.qmp_port)
        qmp2.connect(timeout=10)
        reconnected = True
        print(f"QMP reconnected after {time.monotonic() - reset_start:.1f}s")
        break
    except Exception:
        time.sleep(2)


In [ ]:
# ── Step 5: Reconnect serial and wait for login after reboot ─────────────────
if reconnected:
    sc2 = SerialConsole(port=launcher.serial_port)
    sc2.connect(timeout=30)
    try:
        sc2.wait_for_boot(timeout=180)
        sc2.login(CONSOLE_USER, CONSOLE_PASS)
        post_reset_cpuinfo = sc2.run_command(
            "grep 'processor' /proc/cpuinfo | wc -l", timeout=10
        )
        print(f"CPUs after reboot: {post_reset_cpuinfo.strip()}")
        sc2.close()
    except Exception as e:
        print(f"Post-reboot serial: {e}")
else:
    print("QMP reconnect failed — system_reset may not have completed in time")
    post_reset_cpuinfo = "0"


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert_equal(len(cpus), SMP,
             f"QMP query-cpus-fast returns {SMP} vCPUs",
             action="Check SMP=4 in CONFIG block and -smp 4 in launcher")

assert_equal(cpuinfo.strip(), str(SMP),
             f"Guest /proc/cpuinfo shows {SMP} processors",
             action="SMP secondary CPUs did not come online; check PSCI in dmesg")

cpu_dir_count = len([l for l in cpu_dirs.strip().splitlines() if l.strip()])
assert_equal(cpu_dir_count, SMP,
             f"/sys/devices/system/cpu/ has {SMP} CPU directories",
             action="Sysfs CPU count mismatch — secondaries may be offline")

assert_contains(psci_dmesg, r"[Pp][Ss][Cc][Ii]",
                "PSCI driver registered in dmesg",
                action="Kernel CONFIG_ARM_PSCI_FW not enabled or PSCI not in DT/ACPI")

assert_contains(psci_dmesg, r"HVC|SMC|hvc|smc",
                "PSCI conduit method (HVC or SMC) visible in dmesg",
                action="Check QEMU -machine virt PSCI configuration")

assert_true(reconnected,
            "QMP reconnects after system_reset",
            detail=f"Reconnect took {time.monotonic() - reset_start:.1f}s",
            action="QEMU may not have reset; check -no-reboot flag (remove it for Ch05)")


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| QMP vCPU count | 4 | -smp 4 |
| /proc/cpuinfo processors | 4 | All secondaries online |
| /sys cpu directories | 4 | Sysfs reflects SMP |
| PSCI in dmesg | Present | Driver registered |
| PSCI conduit HVC/SMC | Present | Call path confirmed |
| QMP reconnect post reset | Yes | VM completed reboot |

> **Fidelity gap**: QEMU PSCI uses a thread model (one pthread per vCPU)
> rather than TF-A + SCP firmware. The PSCI CPU_ON sequence is functionally
> identical from the OS perspective; timing and power state transitions
> are not modelled.

## Next Steps
→ **Chapter 6**: SCMI — enumerate clock/power domains from the guest.
